# Custom Quran Audio Processing Pipeline

This notebook processes custom Quran recitation audio files to generate word-level timestamps compatible with the video generation pipeline.

## Workflow Overview

1. **Audio Preprocessing**: Validate and optimize audio format for Whisper
2. **Transcription**: Use OpenAI Whisper to transcribe with word-level timestamps
3. **Ayah Detection**: Match transcribed text to canonical Quran verses using fuzzy matching
4. **Word Alignment**: Align transcribed words to Quran reference words
5. **Timestamp Interpolation**: Fill in missing word timestamps
6. **Validation**: Review and manually correct alignment issues
7. **Export**: Generate Tarteel-compatible JSON for video rendering

## Prerequisites

Install required dependencies:
```bash
pip install openai-whisper pydub rapidfuzz arabic-reshaper python-bidi
```

**Note**: This process requires significant computational resources. For a 10-minute recitation:
- Transcription: ~5-10 minutes on CPU (faster with GPU)
- Alignment: ~1-2 minutes
- Manual review: ~5-10 minutes (depending on accuracy)

## Step 1: Configuration

Set up file paths and processing parameters.

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Path to your audio file
AUDIO_FILE_PATH = "path/to/your/recitation.mp3"  # UPDATE THIS

# Surah and ayah range hints (optional but improves accuracy)
# If you know which surah/ayahs are in the audio, specify here
SURAH_NUMBER = None  # e.g., 1 for Al-Fatiha, or None if unknown
AYA_START = None     # Starting ayah number, or None
AYA_END = None       # Ending ayah number, or None

# Whisper model configuration
# Options: "tiny", "base", "small", "medium", "large"
# Larger models are more accurate but slower
WHISPER_MODEL = "base"  # Recommended: "base" for good balance

# Device configuration (macOS optimized)
# Options: "auto" (recommended), "cpu", "mps", "cuda"
# "auto" will detect Apple Silicon and use Metal acceleration
DEVICE = "auto"  # On M1/M2/M3 Mac, this will automatically use MPS (Metal Performance Shaders)

# Output paths
OUTPUT_DIR = "temp/custom_audio"
OUTPUT_JSON_FILENAME = "custom_recitation.json"

# Processing parameters
CONFIDENCE_THRESHOLD = 0.70  # Minimum confidence for auto-matching (0-1)
NORMALIZE_TA_MARBUTA = False  # Set True for more lenient matching

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  Audio file: {AUDIO_FILE_PATH}")
print(f"  Surah hint: {SURAH_NUMBER if SURAH_NUMBER else 'None (will detect)'}")
print(f"  Whisper model: {WHISPER_MODEL}")
print(f"  Device: {DEVICE}")
print(f"  Output: {OUTPUT_DIR}/{OUTPUT_JSON_FILENAME}")

# Detect if running on macOS
import platform
if platform.system() == "Darwin":
    print(f"\n🍎 macOS detected: {platform.machine()}")
    if "arm" in platform.machine().lower():
        print("   ✓ Apple Silicon detected - MPS acceleration available!")
        print("   ℹ️  Whisper will be ~2-3x faster than CPU")

In [ ]:
# Initialize audio preprocessor
preprocessor = AudioPreprocessor(output_dir=f"{OUTPUT_DIR}/processed_audio")

# Validate audio file
try:
    print("Validating audio file...")
    audio_metadata = preprocessor.validate_audio(AUDIO_FILE_PATH)
    
    print("\n📊 Audio Metadata:")
    for key, value in audio_metadata.items():
        print(f"  {key}: {value}")
    
    # Check if audio needs preprocessing
    needs_preprocessing = (
        audio_metadata['channels'] > 1 or 
        audio_metadata['sample_rate'] != 16000
    )
    
    if needs_preprocessing:
        print("\n🔧 Audio requires preprocessing...")
        processed_audio_path = preprocessor.preprocess(
            AUDIO_FILE_PATH,
            normalize_audio=True
        )
        print(f"✓ Processed audio saved: {processed_audio_path}")
        AUDIO_TO_TRANSCRIBE = processed_audio_path
    else:
        print("\n✓ Audio is already in optimal format")
        AUDIO_TO_TRANSCRIBE = AUDIO_FILE_PATH
        
except Exception as e:
    print(f"❌ Error preprocessing audio: {e}")
    raise

In [ ]:
# Check for existing checkpoint
checkpoint_path = f"{OUTPUT_DIR}/transcription_checkpoint.json"
transcription_result = load_transcription_checkpoint(checkpoint_path)

if transcription_result:
    print("✓ Found existing transcription checkpoint")
    print("  Skipping transcription (delete checkpoint to re-run)")
else:
    # Initialize Whisper transcriber with auto device detection
    print(f"Initializing Whisper model '{WHISPER_MODEL}'...")
    print("(This will download the model on first run)")
    transcriber = WhisperTranscriber(model_name=WHISPER_MODEL, device=DEVICE)
    
    # Transcribe
    print(f"\n🎤 Transcribing audio...")
    print("⏳ This may take several minutes. Please wait...")
    
    try:
        transcription_result = transcriber.transcribe(
            AUDIO_TO_TRANSCRIBE,
            language="ar",
            word_timestamps=True,
            save_json=f"{OUTPUT_DIR}/raw_transcription.json"
        )
        
        # Save checkpoint
        save_transcription_checkpoint(transcription_result, checkpoint_path)
        
        print(f"✓ Transcription complete!")
        print(f"  Full transcription: {transcription_result['text'][:200]}...")
        
    except Exception as e:
        print(f"❌ Transcription failed: {e}")
        raise

# Extract word-level timestamps
print("\n📝 Extracting word timestamps...")
transcriber = WhisperTranscriber(model_name=WHISPER_MODEL, device=DEVICE)
transcribed_words = transcriber.extract_word_timestamps(transcription_result)

print(f"✓ Extracted {len(transcribed_words)} words with timestamps")

In [ ]:
# Display transcribed words in a table
transcription_data = []
for i, word in enumerate(transcribed_words[:50]):  # Show first 50 words
    transcription_data.append({
        "#": i + 1,
        "Word": word.word,
        "Start (s)": f"{word.start:.2f}",
        "End (s)": f"{word.end:.2f}",
        "Duration (s)": f"{word.end - word.start:.2f}",
        "Confidence": f"{word.confidence:.2%}" if word.confidence else "N/A"
    })

df_transcription = pd.DataFrame(transcription_data)
display(HTML("<h4>📝 Transcribed Words (First 50)</h4>"))
display(HTML(df_transcription.to_html(index=False, escape=False)))

if len(transcribed_words) > 50:
    print(f"\n... and {len(transcribed_words) - 50} more words")

# Show full transcribed text
print("\n📄 Full Transcribed Text:")
print("=" * 80)
print(transcription_result['text'])
print("=" * 80)

In [ ]:
# Load Quran text
print("Loading Quran reference text...")
quran_data = load_quran_text("data/quran/quran.json")

# Load translations
with open("data/quran/English wbw translation.json", 'r', encoding='utf-8') as f:
    translation_data = json.load(f)

print(f"✓ Loaded Quran text: {len(quran_data)} surahs")

# Initialize Arabic normalizer
normalizer = ArabicNormalizer(normalize_ta_marbuta=NORMALIZE_TA_MARBUTA)

# Initialize ayah detector
print("Building normalized corpus for matching...")
ayah_detector = AyahDetector(
    quran_data=quran_data,
    normalizer=normalizer,
    confidence_threshold=CONFIDENCE_THRESHOLD
)

print("✓ Quran corpus ready for matching")

In [ ]:
# Detect ayahs from transcription
print("🔍 Detecting ayahs from transcription...")
print(f"Using sliding window approach with overlap...")

detected_ayahs = ayah_detector.detect_ayahs_from_transcription(
    transcribed_words=transcribed_words,
    window_size=15,  # Number of words per window
    overlap=5        # Overlap between windows
)

print(f"\n✓ Detected {len(detected_ayahs)} ayah segments")

# Display detected ayahs
if detected_ayahs:
    detection_data = []
    for i, ayah_info in enumerate(detected_ayahs):
        detection_data.append({
            "#": i + 1,
            "Surah:Ayah": f"{ayah_info['surah']}:{ayah_info['ayah']}",
            "Confidence": f"{ayah_info['confidence']:.1%}",
            "Start": f"{ayah_info['start_time']:.2f}s",
            "End": f"{ayah_info['end_time']:.2f}s",
            "Duration": f"{ayah_info['end_time'] - ayah_info['start_time']:.2f}s",
            "Transcribed": ayah_info['transcribed_text'][:50] + "..."
        })
    
    df_detected = pd.DataFrame(detection_data)
    display(HTML("<h4>🎯 Detected Ayahs</h4>"))
    display(HTML(df_detected.to_html(index=False, escape=False)))
    
    # Flag low-confidence matches
    low_confidence = [d for d in detected_ayahs if d['confidence'] < CONFIDENCE_THRESHOLD]
    if low_confidence:
        print(f"\n⚠️  Warning: {len(low_confidence)} ayahs have low confidence (<{CONFIDENCE_THRESHOLD:.0%})")
        print("   These will require manual review")
else:
    print("❌ No ayahs detected! Check transcription quality or adjust parameters.")

In [ ]:
# Display side-by-side comparison
if detected_ayahs:
    display(HTML("<h4>📋 Transcription vs. Reference Comparison</h4>"))
    
    for i, ayah_info in enumerate(detected_ayahs[:10]):  # Show first 10
        surah = ayah_info['surah']
        ayah = ayah_info['ayah']
        
        # Get reference text
        reference_text = ayah_detector.corpus[(surah, ayah)]['display']
        transcribed_text = ayah_info['transcribed_text']
        
        # Normalize both for comparison
        ref_normalized = normalizer.normalize(reference_text)
        trans_normalized = normalizer.normalize(transcribed_text)
        
        html = f"""
        <div style='border: 1px solid #ddd; padding: 10px; margin: 10px 0; border-radius: 5px;'>
            <h5 style='margin-top: 0;'>Ayah {surah}:{ayah} 
                <span style='color: {"green" if ayah_info["confidence"] >= CONFIDENCE_THRESHOLD else "orange"};'>
                    ({ayah_info['confidence']:.1%} confidence)
                </span>
            </h5>
            <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 15px;'>
                <div>
                    <strong>Reference (Quran):</strong><br>
                    <div style='direction: rtl; font-size: 1.2em; padding: 5px; background: #f9f9f9;'>{reference_text}</div>
                    <small style='color: #666;'>{ref_normalized}</small>
                </div>
                <div>
                    <strong>Transcribed (Whisper):</strong><br>
                    <div style='direction: rtl; font-size: 1.2em; padding: 5px; background: #fff3cd;'>{transcribed_text}</div>
                    <small style='color: #666;'>{trans_normalized}</small>
                </div>
            </div>
        </div>
        """
        display(HTML(html))
    
    if len(detected_ayahs) > 10:
        print(f"\n... and {len(detected_ayahs) - 10} more ayahs")

In [ ]:
# Manual correction example
# Uncomment and modify if you need to fix specific ayahs

# detected_ayahs[0]['surah'] = 1  # Change surah
# detected_ayahs[0]['ayah'] = 2   # Change ayah
# detected_ayahs[0]['confidence'] = 1.0  # Set to 100% if manually verified

# You can also remove incorrect detections:
# detected_ayahs = [d for i, d in enumerate(detected_ayahs) if i != 0]  # Remove first

# Or add missing ayahs:
# detected_ayahs.insert(0, {
#     'surah': 1,
#     'ayah': 1,
#     'confidence': 1.0,
#     'start_time': 0.0,
#     'end_time': 5.0,
#     'transcribed_text': '...',
#     'word_indices': (0, 4)
# })

print(f"Current ayah count: {len(detected_ayahs)}")
print("Edit the cell above to make manual corrections, then re-run")

In [ ]:
# Initialize word aligner
word_aligner = WordAligner(normalizer=normalizer)

# Align words for each detected ayah
print("🔗 Aligning words to Quran reference...")
aligned_ayahs = []

for ayah_info in detected_ayahs:
    surah = ayah_info['surah']
    ayah = ayah_info['ayah']
    
    # Get transcribed words for this ayah
    word_start_idx, word_end_idx = ayah_info['word_indices']
    ayah_transcribed_words = transcribed_words[word_start_idx:word_end_idx]
    
    # Get reference words
    reference_words = ayah_detector.corpus[(surah, ayah)]['words']
    
    # Align words
    alignments = word_aligner.align_words(
        transcribed_words=ayah_transcribed_words,
        reference_words=reference_words,
        surah=surah,
        ayah=ayah
    )
    
    # Interpolate missing words
    all_word_times = word_aligner.interpolate_missing_words(
        alignments=alignments,
        total_reference_words=len(reference_words),
        ayah_start_time=ayah_info['start_time'],
        ayah_end_time=ayah_info['end_time']
    )
    
    # Create AyahMatch object
    from audio_processing_utils import AyahMatch
    
    aligned_ayah = AyahMatch(
        surah=surah,
        ayah=ayah,
        confidence=ayah_info['confidence'],
        transcribed_text=ayah_info['transcribed_text'],
        reference_text=ayah_detector.corpus[(surah, ayah)]['display'],
        word_alignments=alignments
    )
    
    # Store complete alignment with all words
    aligned_ayah.complete_word_times = all_word_times
    aligned_ayahs.append(aligned_ayah)

print(f"✓ Aligned {len(aligned_ayahs)} ayahs with word-level timestamps")

In [ ]:
# Display word alignment for first few ayahs
display(HTML("<h4>🔍 Word-Level Alignment Details</h4>"))

for aligned_ayah in aligned_ayahs[:3]:  # Show first 3 ayahs
    surah = aligned_ayah.surah
    ayah = aligned_ayah.ayah
    
    print(f"\n{'='*80}")
    print(f"Ayah {surah}:{ayah} - {aligned_ayah.confidence:.1%} confidence")
    print(f"{'='*80}")
    
    # Get reference words
    reference_words = ayah_detector.corpus[(surah, ayah)]['words']
    
    # Create alignment table
    alignment_data = []
    for word_pos, start_time, end_time in aligned_ayah.complete_word_times:
        word_text = reference_words[word_pos - 1] if word_pos <= len(reference_words) else "N/A"
        
        # Check if this word was directly transcribed
        directly_transcribed = any(
            pos == word_pos for _, pos in aligned_ayah.word_alignments
        )
        
        # Get translation
        trans_key = f"{surah}:{ayah}:{word_pos}"
        translation = translation_data.get(trans_key, "")
        
        alignment_data.append({
            "Pos": word_pos,
            "Arabic": word_text,
            "Translation": translation[:30] + "..." if len(translation) > 30 else translation,
            "Start": f"{start_time:.2f}s",
            "End": f"{end_time:.2f}s",
            "Duration": f"{end_time - start_time:.2f}s",
            "Source": "✓ Whisper" if directly_transcribed else "⚠ Interpolated"
        })
    
    df_alignment = pd.DataFrame(alignment_data)
    display(HTML(df_alignment.to_html(index=False, escape=False)))
    
    # Show statistics
    whisper_count = sum(1 for _, pos in aligned_ayah.word_alignments)
    interpolated_count = len(reference_words) - whisper_count
    print(f"\nStatistics:")
    print(f"  Total words: {len(reference_words)}")
    print(f"  Directly transcribed: {whisper_count} ({whisper_count/len(reference_words)*100:.1f}%)")
    print(f"  Interpolated: {interpolated_count} ({interpolated_count/len(reference_words)*100:.1f}%)")

if len(aligned_ayahs) > 3:
    print(f"\n... and {len(aligned_ayahs) - 3} more ayahs")

In [ ]:
# Build Tarteel-compatible JSON structure
tarteel_data = {}

for aligned_ayah in aligned_ayahs:
    key = f"{aligned_ayah.surah}:{aligned_ayah.ayah}"
    
    # Convert word times to segments format: [[word_position, start_ms, end_ms]]
    segments = []
    for word_pos, start_time, end_time in aligned_ayah.complete_word_times:
        segments.append([
            word_pos,
            int(start_time * 1000),  # Convert to milliseconds
            int(end_time * 1000)
        ])
    
    # Calculate duration (last word end time)
    duration = max(seg[2] for seg in segments) if segments else 0
    
    tarteel_data[key] = {
        "surah_number": aligned_ayah.surah,
        "ayah_number": aligned_ayah.ayah,
        "audio_url": os.path.abspath(AUDIO_FILE_PATH),  # Use absolute path
        "duration": duration,
        "segments": segments,
        "metadata": {
            "confidence": aligned_ayah.confidence,
            "source": "custom_whisper_transcription",
            "model": WHISPER_MODEL
        }
    }

# Save to file
output_json_path = os.path.join(OUTPUT_DIR, OUTPUT_JSON_FILENAME)
with open(output_json_path, 'w', encoding='utf-8') as f:
    json.dump(tarteel_data, f, ensure_ascii=False, indent=4)

print(f"✅ Tarteel-format JSON saved to:")
print(f"   {output_json_path}")
print(f"\n📊 Export Summary:")
print(f"   Total ayahs: {len(tarteel_data)}")
print(f"   Total words: {sum(len(data['segments']) for data in tarteel_data.values())}")
print(f"   Average confidence: {sum(aligned_ayah.confidence for aligned_ayah in aligned_ayahs) / len(aligned_ayahs):.1%}")

# Display sample of exported data
print(f"\n📄 Sample JSON structure:")
sample_key = list(tarteel_data.keys())[0]
sample_data = tarteel_data[sample_key].copy()
sample_data['segments'] = sample_data['segments'][:3] + ['...']
print(json.dumps({sample_key: sample_data}, ensure_ascii=False, indent=2))

In [ ]:
# Instructions for using this JSON with video_gen.ipynb
print("🎬 Next Steps: Generate Video")
print("=" * 80)
print("\n1. Open video_gen.ipynb")
print("\n2. Update the Reciter enum in quran_utils.py to add your custom reciter:")
print("   ```python")
print("   class Reciter(Enum):")
print("       # ... existing reciters ...")
print(f"       CUSTOM = '{output_json_path}'")
print("   ```")
print("\n3. In video_gen.ipynb, use your custom reciter:")
print("   ```python")
print("   reciter = Reciter.CUSTOM")
print("   ```")
print("\n4. Set the surah and ayah range to match your audio:")
if aligned_ayahs:
    first_ayah = aligned_ayahs[0]
    last_ayah = aligned_ayahs[-1]
    print(f"   ```python")
    print(f"   surah_number = {first_ayah.surah}")
    print(f"   aya_start = {first_ayah.ayah}")
    print(f"   aya_end = {last_ayah.ayah}")
    print(f"   ```")
print("\n5. Run video_gen.ipynb cells sequentially")
print("\n⚠️  Important: Review the generated video to ensure:")
print("   - Word timing is synchronized correctly")
print("   - All ayahs are present")
print("   - Background videos are appropriate")
print("\n" + "=" * 80)

In [ ]:
# Uncomment to clean up temporary files
# import shutil

# try:
#     # Remove processed audio
#     if os.path.exists(f"{OUTPUT_DIR}/processed_audio"):
#         shutil.rmtree(f"{OUTPUT_DIR}/processed_audio")
#         print("✓ Removed processed audio files")
#     
#     # Remove transcription checkpoint (if you want to re-run from scratch)
#     # if os.path.exists(checkpoint_path):
#     #     os.remove(checkpoint_path)
#     #     print("✓ Removed transcription checkpoint")
#     
#     # Keep the final JSON and raw transcription for reference
#     print(f"\n✓ Keeping:")
#     print(f"  - {output_json_path}")
#     print(f"  - {OUTPUT_DIR}/raw_transcription.json")
#     
# except Exception as e:
#     print(f"Error during cleanup: {e}")

print("💾 Files preserved for future use")
print(f"   JSON: {output_json_path}")
print(f"   Raw transcription: {OUTPUT_DIR}/raw_transcription.json")

### Common Issues and Solutions

**1. Low Ayah Detection Confidence (<70%)**
- **Cause**: Whisper transcription errors, audio quality issues, or background noise
- **Solutions**:
  - Use a larger Whisper model (`medium` or `large`)
  - Clean audio to remove background noise
  - Manually correct misdetected ayahs in Step 5
  - Lower `CONFIDENCE_THRESHOLD` (but review results carefully)

**2. No Ayahs Detected**
- **Cause**: Audio language not set correctly, or audio contains non-Quranic content
- **Solutions**:
  - Verify audio is Quran recitation
  - Check if `SURAH_NUMBER` hint is correct
  - Review raw transcription in Step 3 to see what Whisper detected
  - Try with `language="ar"` explicitly set

**3. Word Timestamps Misaligned**
- **Cause**: Whisper missing words, or reciter has long pauses/elongations
- **Solutions**:
  - Most interpolated words will still be close - review in Step 6
  - Manually adjust timestamps in exported JSON if needed
  - Use audio from professional reciters for best results

**4. Transcription Taking Too Long**
- **Cause**: Large audio file or slow CPU
- **Solutions**:
  - Use smaller Whisper model (`tiny` or `base`)
  - Split audio into smaller segments (by surah/ayah if known)
  - Use GPU acceleration (change `device="cuda"`)
  - Resume from checkpoint if interrupted

**5. Memory Errors**
- **Cause**: Large audio files exhausting RAM
- **Solutions**:
  - Process audio in chunks (split by surah)
  - Use smaller Whisper model
  - Close other applications
  - Use a machine with more RAM

**6. Arabic Text Not Displaying Correctly**
- **Cause**: Missing font or encoding issues
- **Solutions**:
  - Check browser supports Arabic Unicode (RTL)
  - Verify JSON files have `utf-8` encoding
  - Try different viewing environment

### Best Practices

1. **Start with short audio clips** (1-2 ayahs) to test the pipeline
2. **Use high-quality audio** (clear recitation, minimal background noise)
3. **Provide surah/ayah hints** when known to improve accuracy
4. **Always review alignment** before video generation
5. **Keep raw transcription files** for debugging
6. **Document manual corrections** for future reference

### Performance Benchmarks

Typical processing times on M1 MacBook Pro:

| Audio Length | Whisper Model | Transcription | Alignment | Total   |
|--------------|---------------|---------------|-----------|---------|
| 1 minute     | base          | ~30s          | ~5s       | ~35s    |
| 5 minutes    | base          | ~2.5min       | ~15s      | ~3min   |
| 10 minutes   | base          | ~5min         | ~30s      | ~6min   |
| 10 minutes   | large         | ~15min        | ~30s      | ~16min  |

*Times vary based on hardware and audio complexity*

---

## Troubleshooting Guide

Common issues and solutions when processing custom audio.

## Cleanup (Optional)

Remove temporary files to free up space.

## Step 8: Integration with Video Generation

Use the generated JSON file with `video_gen.ipynb` to create videos.

## Step 7: Export to Tarteel Format

Convert the aligned data to Tarteel-compatible JSON format for video generation.

### View Word-Level Alignment

Display detailed word-by-word alignment for validation.

## Step 6: Word-Level Alignment

Align transcribed words to canonical Quran words with timestamp mapping.

### Manual Correction (Optional)

If any ayahs are misdetected, you can manually correct them here.

### Compare Transcription vs Reference

View transcribed text alongside canonical Quran text for validation.

## Step 5: Detect Ayahs from Transcription

Match transcribed text segments to Quran ayahs using fuzzy matching.

## Step 4: Load Quran Reference Text

Load the canonical Quran text for matching against transcription.

### View Transcription Results

Display the raw transcription with word-level timestamps.

## Step 3: Whisper Transcription

Transcribe the audio using OpenAI Whisper with word-level timestamps.

**Note**: This step can take several minutes depending on audio length and model size.

## Step 2: Audio Preprocessing

Validate and preprocess the audio file for optimal Whisper transcription.

In [ ]:
# Import required modules
import os
import json
from pathlib import Path
from typing import List, Dict, Optional
import pandas as pd
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Import custom processing modules
from audio_processing_utils import (
    WhisperTranscriber,
    AudioPreprocessor,
    ArabicNormalizer,
    TranscribedWord,
    load_quran_text,
    save_transcription_checkpoint,
    load_transcription_checkpoint
)

from alignment_utils import (
    AyahDetector,
    WordAligner,
    convert_to_tarteel_format
)

print("✓ All modules imported successfully")